In [ ]:
import pyspark
print(pyspark.__version__)

4.0.2


**Create Spark Session**

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName("NYC_Taxi_Analytics")\
.getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [ ]:
spark

**Download Dataset and check**

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet

--2026-06-02 05:10:43--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.222, 54.230.248.205, 54.230.248.73, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.222|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59158238 (56M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-01.parquet’

yellow_tripdata_202 100%[===================>]  56.42M   247MB/s    in 0.2s    

2026-06-02 05:10:44 (247 MB/s) - ‘yellow_tripdata_2025-01.parquet’ saved [59158238/59158238]



In [ ]:
!ls

sample_data  yellow_tripdata_2025-01.parquet


**Read Dataset with Spark & Verify**

In [ ]:
df = spark.read.parquet(
    "yellow_tripdata_2025-01.parquet"
)

In [ ]:
df.show(5) # by default shows first 20 rows

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [ ]:
df.count()

3475226

In [ ]:
len(df.columns) ## Count number of columns

20

In [ ]:
df.printSchema() ## View dataset schema

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## **Display Sample Records**

In [ ]:
df.show(10, truncate = False) ## by default truncate is true

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2025-01-01 00:18:38 |2025-01-01 00:26:59  |1              |1.6          |1         |N                 |229         |237 

## **Count rows and columns**



In [ ]:
df.count() ## count rows

3475226

In [ ]:
len(df.columns) ## count columns

20

In [ ]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

## **Summary Statistics**

In [ ]:
df.describe().show()

+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+-------------------+
|summary|           VendorID|   passenger_count|     trip_distance|        RatecodeID|store_and_fwd_flag|     PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee| cbd_congestion_fee|
+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+-

In [ ]:
print("Rows: ", df.count())
print("Columns: ", len(df.columns))

Rows:  3475226
Columns:  20


# **Data Cleaning**

### **Check Missing Values**

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.functions import count
from pyspark.sql.functions import when

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)

    for c in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|         540149|            0|    540149|            540149|           0|    

## **Count of Missing Values --- Column-wise**

passenger_count --> 540149

RatecodeID --> 540149

store_and_fwd_flag -> 540149

congestion_surcharge --> 540149

Airport_fee --> 540149

### **Check duplicate records**

In [ ]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print(f"Total Rows: ", total_rows)
print(f"Unique Records: ", unique_rows)
print(f"Duplicate Records: ", total_rows - unique_rows)

Total Rows:  3475226
Unique Records:  3475226
Duplicate Records:  0


### **Check negative trip distance**

In [ ]:
df.filter(
    col('trip_distance') < 0
).count()

0

### **Check negative fare amount**

In [ ]:
df.filter(
    col('fare_amount') < 0
).count()

144118

### **Check negative total amount**

In [ ]:
df.filter(
    col('total_amount') < 0
).count()

63037

### **Check passenger count issues**

In [ ]:
df.filter(
    col('passenger_count') <= 0
).count()

24656

## **Summary Report**


In [ ]:
print(f"Total Rows: {df.count()}" )
print(f"Duplicates: {total_rows - unique_rows}")

print(f"Negative Distance: {df.filter(col('trip_distance')<0).count()}")
print(f"Negative Fare: {df.filter(col('fare_amount')<0).count()}")
print(f"Negative Total Amount: {df.filter(col('total_amount')<0).count()}")
print(f"Invalid Passenger: {df.filter(col('passenger_count')<=0).count()}")

Total Rows: 3475226
Duplicates: 0
Negative Distance: 0
Negative Fare: 144118
Negative Total Amount: 63037
Invalid Passenger: 24656


In [ ]:
df.show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

### **Negative Fare Amount**

In [ ]:
df.filter(
    col('fare_amount') < 0)\
    .select(
        'fare_amount',
        'total_amount',
        'payment_type'
    )\
    .show(20, False)


+-----------+------------+------------+
|fare_amount|total_amount|payment_type|
+-----------+------------+------------+
|-7.2       |-8.54       |2           |
|-6.5       |-11.5       |4           |
|-16.3      |-21.3       |4           |
|-12.1      |-17.1       |2           |
|-7.2       |-12.2       |4           |
|-14.2      |-19.2       |4           |
|-3.0       |-5.5        |4           |
|-6.5       |-11.5       |2           |
|-24.0      |-29.0       |2           |
|-6.5       |-11.5       |2           |
|-4.4       |-9.4        |4           |
|-111.5     |-123.44     |3           |
|-38.7      |-43.7       |2           |
|-32.4      |-48.59      |4           |
|-10.0      |-15.0       |3           |
|-3.7       |-8.7        |2           |
|-12.8      |-17.8       |4           |
|-66.0      |-70.25      |4           |
|-10.0      |-15.0       |4           |
|-4.4       |-9.4        |4           |
+-----------+------------+------------+
only showing top 20 rows


### **Negative Total Amount**

In [ ]:
df.filter(col("total_amount") < 0) \
  .select(
      "fare_amount",
      "tip_amount",
      "total_amount"
  ) \
  .show(20, False)

+-----------+----------+------------+
|fare_amount|tip_amount|total_amount|
+-----------+----------+------------+
|-7.2       |3.66      |-8.54       |
|-6.5       |0.0       |-11.5       |
|-16.3      |0.0       |-21.3       |
|-12.1      |0.0       |-17.1       |
|-7.2       |0.0       |-12.2       |
|-14.2      |0.0       |-19.2       |
|-3.0       |0.0       |-5.5        |
|-6.5       |0.0       |-11.5       |
|-24.0      |0.0       |-29.0       |
|-6.5       |0.0       |-11.5       |
|-4.4       |0.0       |-9.4        |
|-111.5     |0.0       |-123.44     |
|-38.7      |0.0       |-43.7       |
|-32.4      |0.0       |-48.59      |
|-10.0      |0.0       |-15.0       |
|-3.7       |0.0       |-8.7        |
|-12.8      |0.0       |-17.8       |
|-66.0      |0.0       |-70.25      |
|-10.0      |0.0       |-15.0       |
|-4.4       |0.0       |-9.4        |
+-----------+----------+------------+
only showing top 20 rows


### **Invalid Passenger Count**

In [ ]:
df.groupBy('passenger_count')\
.count()\
.orderBy('passenger_count')\
.show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL| 540149|
|              0|  24656|
|              1|2322434|
|              2| 407761|
|              3|  91409|
|              4|  59009|
|              5|  17786|
|              6|  12004|
|              7|      4|
|              8|     11|
|              9|      3|
+---------------+-------+



In [ ]:
df.filter(col("fare_amount") < 0) \
  .select(
      "fare_amount",
      "total_amount",
      "trip_distance"
  ) \
  .show(10, False)

+-----------+------------+-------------+
|fare_amount|total_amount|trip_distance|
+-----------+------------+-------------+
|-7.2       |-8.54       |0.71         |
|-6.5       |-11.5       |0.69         |
|-16.3      |-21.3       |0.97         |
|-12.1      |-17.1       |1.42         |
|-7.2       |-12.2       |0.6          |
|-14.2      |-19.2       |1.88         |
|-3.0       |-5.5        |0.01         |
|-6.5       |-11.5       |0.6          |
|-24.0      |-29.0       |3.84         |
|-6.5       |-11.5       |0.92         |
+-----------+------------+-------------+
only showing top 10 rows


In [ ]:
df.filter(col("total_amount") < 0) \
  .select(
      "fare_amount",
      "tip_amount",
      "total_amount"
  ) \
  .show(10, False)

+-----------+----------+------------+
|fare_amount|tip_amount|total_amount|
+-----------+----------+------------+
|-7.2       |3.66      |-8.54       |
|-6.5       |0.0       |-11.5       |
|-16.3      |0.0       |-21.3       |
|-12.1      |0.0       |-17.1       |
|-7.2       |0.0       |-12.2       |
|-14.2      |0.0       |-19.2       |
|-3.0       |0.0       |-5.5        |
|-6.5       |0.0       |-11.5       |
|-24.0      |0.0       |-29.0       |
|-6.5       |0.0       |-11.5       |
+-----------+----------+------------+
only showing top 10 rows


In [ ]:
df.groupBy("passenger_count") \
  .count() \
  .orderBy("passenger_count") \
  .show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL| 540149|
|              0|  24656|
|              1|2322434|
|              2| 407761|
|              3|  91409|
|              4|  59009|
|              5|  17786|
|              6|  12004|
|              7|      4|
|              8|     11|
|              9|      3|
+---------------+-------+



### **Removing Negative Revenue records**

In [ ]:
from pyspark.sql.functions import col

clean_df = df.filter(
    (col('fare_amount') >= 0) &
    (col('total_amount') >= 0)
)

# clean_df.show()

### **Replacing Missing Passenger Counts**

In [ ]:
clean_df = clean_df.fillna(
    {'passenger_count' : 0}
)

## **Check Negative Fare after cleaning**

In [ ]:
clean_df.filter(
    col('fare_amount') < 0
).count()

0

## **Check Negative Total after cleaning**

In [ ]:
clean_df.filter(
    col('total_amount') < 0
 ).count()

0

## **Check Missing Passenger count after cleaning**

In [ ]:
clean_df.filter(
    col('passenger_count').isNull()
).count()

0

## **Comparing datasize count after cleaning**

In [ ]:
print(f"Dataset Count Before Cleaning: {df.count()}")
print(f"Dataset Count After Cleaning: {clean_df.count()}")

print(f"Rows removed during cleaning: {df.count() - clean_df.count()}")


Dataset Count Before Cleaning: 3475226
Dataset Count After Cleaning: 3330787
Rows removed during cleaning: 144439


# **Feature Engineering**

### **Create Trip Duration**


How long does each trip take?

In [ ]:
from pyspark.sql.functions import *

clean_df = clean_df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp('tpep_dropoff_datetime') \
        - unix_timestamp('tpep_pickup_datetime')

    ) / 60
)

Create pickup hour

In [ ]:
clean_df = clean_df.withColumn(
    "pickup_hour",
    hour('tpep_pickup_datetime')
)

Create pickup day

In [ ]:
clean_df = clean_df.withColumn(
    "pickup_day",
    dayofmonth('tpep_pickup_datetime')
)

Create pickup month

In [ ]:
clean_df = clean_df.withColumn(
    'pickup_month',
    month('tpep_pickup_datetime')
)

Create pickup weekday

In [ ]:
clean_df = clean_df.withColumn(
    "pickup_weekday",
    date_format(
        'tpep_pickup_datetime',
        "EEEE"
    )
)

Create Tip Passenger

In [ ]:
clean_df = clean_df.withColumn(
    "tip_percentage",
    when(
        col("fare_amount") > 0,
        round(
            (col("tip_amount")
            / col("fare_amount")) * 100,
            2
        )
    ).otherwise(0)
)

## round(value, 2)

### **Verify New Columns**

In [ ]:
clean_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "pickup_weekday",
    "tip_percentage"
).show(10, False)

+--------------------+---------------------+---------------------+-----------+----------+------------+--------------+--------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|pickup_hour|pickup_day|pickup_month|pickup_weekday|tip_percentage|
+--------------------+---------------------+---------------------+-----------+----------+------------+--------------+--------------+
|2025-01-01 00:18:38 |2025-01-01 00:26:59  |8.35                 |0          |1         |1           |Wednesday     |30.0          |
|2025-01-01 00:32:40 |2025-01-01 00:35:13  |2.55                 |0          |1         |1           |Wednesday     |39.61         |
|2025-01-01 00:44:04 |2025-01-01 00:46:01  |1.95                 |0          |1         |1           |Wednesday     |39.22         |
|2025-01-01 00:14:27 |2025-01-01 00:20:01  |5.566666666666666    |0          |1         |1           |Wednesday     |0.0           |
|2025-01-01 00:21:34 |2025-01-01 00:25:06  |3.533333333333333    |0  

## **Quick Validation Checks**

**Check Trip Duration**

In [ ]:
clean_df.select(
    min('trip_duration_minutes'),
    max('trip_duration_minutes'),
    avg('trip_duration_minutes')
).show()

+--------------------------+--------------------------+--------------------------+
|min(trip_duration_minutes)|max(trip_duration_minutes)|avg(trip_duration_minutes)|
+--------------------------+--------------------------+--------------------------+
|       -51472.316666666666|         5626.316666666667|        15.037044188054935|
+--------------------------+--------------------------+--------------------------+



**Check Pickup Hour**

In [ ]:
clean_df.select(
    min('pickup_hour'),
    max('pickup_hour')
).show()

+----------------+----------------+
|min(pickup_hour)|max(pickup_hour)|
+----------------+----------------+
|               0|              23|
+----------------+----------------+



**Check Tip Percentage**

In [ ]:
clean_df.select(
    min('tip_percentage'),
    max('tip_percentage')
).show()

+-------------------+-------------------+
|min(tip_percentage)|max(tip_percentage)|
+-------------------+-------------------+
|                0.0|           400000.0|
+-------------------+-------------------+



# **Exploratory Data Analysis**

**1. Total Revenue**

In [ ]:
from pyspark.sql.functions import sum, format_number

clean_df.agg(
    format_number(sum('total_amount')\
                  ,2).alias('total_revenue'))\
                  .show(truncate = False)

## $90.34 million generated approximately

+-------------+
|total_revenue|
+-------------+
|90,336,320.10|
+-------------+



**2. Total Trips**

In [ ]:
print(f"Total Trips: {clean_df.count()}")

## 3.3 million trips were completed

Total Trips: 3330787


**3. Average Fare**

In [ ]:
clean_df.agg(
    avg('fare_amount').alias('average_fare')
).show()

## $18.32 is average fare

+------------------+
|      average_fare|
+------------------+
|18.325804529082852|
+------------------+



**4. Average Trip Distance**

In [ ]:
clean_df.agg(
    avg('trip_distance').alias('avg_distance')
).show()

## 5.39 miles travelled approximately

+------------------+
|      avg_distance|
+------------------+
|5.3872118391231485|
+------------------+



**5. Average Trip Duration**

In [ ]:
clean_df.agg(
    avg("trip_duration_minutes").alias("avg_duration")
).show()

## trips lasted "15.03 minutes" approximately

+------------------+
|      avg_duration|
+------------------+
|15.037044188054935|
+------------------+



**6. Payment Method Distribution**

In [ ]:
clean_df.groupBy('payment_type')\
.count()\
.orderBy(col('count').desc())\
.show()

## 1 refer credit card payment
## Credit card payments dominates

+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2444378|
|           0| 455327|
|           2| 376318|
|           4|  39070|
|           3|  15693|
|           5|      1|
+------------+-------+



**7. Trips by hour**

In [ ]:
clean_df.groupBy('pickup_hour')\
.count()\
.orderBy('pickup_hour')\
.show()


## 4 - 7 PM was the busiest period

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0| 87820|
|          1| 60540|
|          2| 41045|
|          3| 26260|
|          4| 18527|
|          5| 21037|
|          6| 47355|
|          7| 97937|
|          8|135421|
|          9|138889|
|         10|144706|
|         11|156189|
|         12|170631|
|         13|180774|
|         14|196333|
|         15|207161|
|         16|209883|
|         17|242278|
|         18|253993|
|         19|210807|
+-----------+------+
only showing top 20 rows


**8. Trips by weekday**

In [ ]:
from contextlib import AsyncExitStack
clean_df.groupBy('pickup_weekday')\
.count()\
.orderBy('count', ascending = False)\
.show()

## Thursday's generated highest trip volume

+--------------+------+
|pickup_weekday| count|
+--------------+------+
|      Thursday|590022|
|        Friday|562245|
|     Wednesday|550355|
|      Saturday|456642|
|       Tuesday|437245|
|        Sunday|383990|
|        Monday|350288|
+--------------+------+



# **Revenue and KPI Analysis**

**1. Which hour generates more revenue?**

In [ ]:
revenue_by_hour = clean_df.groupBy('pickup_hour')\
.agg(
    round(
        sum('total_amount'),2).alias('total_revenue'))\
        .orderBy('pickup_hour', ascending = False)

revenue_by_hour.show()

+-----------+-------------+
|pickup_hour|total_revenue|
+-----------+-------------+
|         23|   3594703.76|
|         22|   4681730.55|
|         21|   5264911.75|
|         20|   4999636.19|
|         19|   5655233.37|
|         18|   6639354.65|
|         17|   6655982.19|
|         16|   6136184.58|
|         15|   5671291.87|
|         14|   5358862.61|
|         13|   4801315.55|
|         12|   5281467.73|
|         11|    3985369.1|
|         10|   3730676.52|
|          9|   3532316.66|
|          8|   3428509.19|
|          7|   2621584.77|
|          6|   1380396.15|
|          5|    734579.52|
|          4|    574666.13|
+-----------+-------------+
only showing top 20 rows


**2. Top Revenue Hours**

In [ ]:
revenue_by_hour.orderBy(
    col("total_revenue").desc()
).show(10)

+-----------+-------------+
|pickup_hour|total_revenue|
+-----------+-------------+
|         17|   6655982.19|
|         18|   6639354.65|
|         16|   6136184.58|
|         15|   5671291.87|
|         19|   5655233.37|
|         14|   5358862.61|
|         12|   5281467.73|
|         21|   5264911.75|
|         20|   4999636.19|
|         13|   4801315.55|
+-----------+-------------+
only showing top 10 rows


**3. Revenue by Weekday**

In [ ]:
revenue_by_weekday = clean_df.groupBy(
    "pickup_weekday"
).agg(
    round(sum('total_amount'),2).alias('total_revenue')
)

revenue_by_weekday.show(truncate = False)

+--------------+-------------+
|pickup_weekday|total_revenue|
+--------------+-------------+
|Wednesday     |1.4729743E7  |
|Tuesday       |1.167045425E7|
|Friday        |1.520853842E7|
|Thursday      |1.608670233E7|
|Saturday      |1.154542823E7|
|Monday        |1.073964766E7|
|Sunday        |1.035580621E7|
+--------------+-------------+



**4. Average Revenue Per Trip**

In [ ]:
clean_df.agg(
    round(avg('total_amount'), 2).alias('avg_revenue_by_trip')
).show()

+-------------------+
|avg_revenue_by_trip|
+-------------------+
|              27.12|
+-------------------+



**5. Revenue Contribution by Payment Type**

In [ ]:
payment_revenue = clean_df.groupBy(
    'payment_type'
).agg(
    round(sum('total_amount') ,2).alias('revenue')
)

payment_revenue.show()

+------------+------------+
|payment_type|     revenue|
+------------+------------+
|           5|         0.0|
|           1|6.86370759E7|
|           3|   346446.92|
|           2|  8884569.01|
|           4|  1923300.97|
|           0|1.05449273E7|
+------------+------------+



**6. Peak Revenue Weekdays**

In [ ]:
revenue_by_weekday.orderBy(
    col('total_revenue').desc()
).show()

+--------------+-------------+
|pickup_weekday|total_revenue|
+--------------+-------------+
|      Thursday|1.608670233E7|
|        Friday|1.520853842E7|
|     Wednesday|  1.4729743E7|
|       Tuesday|1.167045425E7|
|      Saturday|1.154542823E7|
|        Monday|1.073964766E7|
|        Sunday|1.035580621E7|
+--------------+-------------+



**7. Revenue per Mile**

In [ ]:
clean_df = clean_df.withColumn(
    "revenue_per_mile",
    when(
        col("trip_distance") > 0,

        round(
            col("total_amount")
            /
            col("trip_distance"),
            2
        )

    ).otherwise(0)
)

**8. Creation of KPI Summary**

In [ ]:
kpi_summary = clean_df.agg(

    round(
        sum("total_amount"),
        2
    ).alias("Total Revenue"),

    count("*").alias("Total Trips"),

    round(
        avg("fare_amount"),
        2
    ).alias("Average Fare"),

    round(
        avg("trip_distance"),
        2
    ).alias("Average Distance"),

    round(
        avg("trip_duration_minutes"),
        2
    ).alias("Average Duration")

)

kpi_summary.show()

+-------------+-----------+------------+----------------+----------------+
|Total Revenue|Total Trips|Average Fare|Average Distance|Average Duration|
+-------------+-----------+------------+----------------+----------------+
| 9.03363201E7|    3330787|       18.33|            5.39|           15.04|
+-------------+-----------+------------+----------------+----------------+



In [ ]:
# Revenue by Hour
revenue_by_hour.show()

# Top Revenue Hours
revenue_by_hour.orderBy(
    col("total_revenue").desc()
).show(10)

# Revenue by Weekday
revenue_by_weekday.show(truncate=False)

# Revenue by Payment Type
payment_revenue.show()

# KPI Summary
kpi_summary.show()

+-----------+-------------+
|pickup_hour|total_revenue|
+-----------+-------------+
|         23|   3594703.76|
|         22|   4681730.55|
|         21|   5264911.75|
|         20|   4999636.19|
|         19|   5655233.37|
|         18|   6639354.65|
|         17|   6655982.19|
|         16|   6136184.58|
|         15|   5671291.87|
|         14|   5358862.61|
|         13|   4801315.55|
|         12|   5281467.73|
|         11|    3985369.1|
|         10|   3730676.52|
|          9|   3532316.66|
|          8|   3428509.19|
|          7|   2621584.77|
|          6|   1380396.15|
|          5|    734579.52|
|          4|    574666.13|
+-----------+-------------+
only showing top 20 rows
+-----------+-------------+
|pickup_hour|total_revenue|
+-----------+-------------+
|         17|   6655982.19|
|         18|   6639354.65|
|         16|   6136184.58|
|         15|   5671291.87|
|         19|   5655233.37|
|         14|   5358862.61|
|         12|   5281467.73|
|         21|   5264911

# **Customer Behaviour Analysis**

**1. Payment Method Distribution(%)**

In [ ]:
from pyspark.sql.functions import count

total_trips = clean_df.count()

payment_distribution = clean_df.groupBy(
    'payment_type'
).agg(
    count('*').alias('trip_count')
)

payment_distribution = payment_distribution.withColumn(
    'percentage',
    round((col('trip_count') / total_trips) * 100, 2)
)

payment_distribution.show()

+------------+----------+----------+
|payment_type|trip_count|percentage|
+------------+----------+----------+
|           5|         1|       0.0|
|           1|   2444378|     73.39|
|           3|     15693|      0.47|
|           2|    376318|      11.3|
|           4|     39070|      1.17|
|           0|    455327|     13.67|
+------------+----------+----------+



**2. Average Tip Amount**

In [ ]:
clean_df.agg(
    round(avg('tip_amount'), 2).alias('avg_tip')
).show()

+-------+
|avg_tip|
+-------+
|   3.09|
+-------+



**3. Average Tip Percentage**

In [ ]:
clean_df.agg(
    round(avg('tip_percentage'), 2).alias('avg_tip_percentage')
).show()

+------------------+
|avg_tip_percentage|
+------------------+
|              20.0|
+------------------+



**4. Top Tipping Hours**

In [ ]:
clean_df.groupBy(
    'pickup_hour'
).agg(
    round(avg('tip_percentage'), 2).alias('avg_tip_percentage')
).orderBy(
    col('avg_tip_percentage').desc()
).show()

+-----------+------------------+
|pickup_hour|avg_tip_percentage|
+-----------+------------------+
|         18|              21.9|
|         15|             21.77|
|         16|             21.69|
|         17|             21.68|
|         19|             21.02|
|         14|             20.84|
|         20|             20.49|
|         10|             20.24|
|         11|             20.13|
|         12|             20.08|
|         21|             20.01|
|         13|             19.97|
|          9|             19.64|
|         23|             18.65|
|         22|             18.56|
|          8|             17.68|
|          3|              17.6|
|          0|             17.21|
|          1|              17.1|
|          2|             17.04|
+-----------+------------------+
only showing top 20 rows


**5. Passenger Count Distrbution**

In [ ]:
clean_df.groupBy(
    'passenger_count'
).count()\
.orderBy(
    'passenger_count'
).show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|              0| 479978|
|              1|2277734|
|              2| 397726|
|              3|  89056|
|              4|  56591|
|              5|  17719|
|              6|  11965|
|              7|      4|
|              8|     11|
|              9|      3|
+---------------+-------+



**6. Revenue by Passenger Count**

In [ ]:
clean_df.groupBy(
    'passenger_count'
).agg(
    round(sum('total_amount'), 2).alias('revenue')
).orderBy(
    col('revenue').desc()
).show()

+---------------+-------------+
|passenger_count|      revenue|
+---------------+-------------+
|              1|6.181546996E7|
|              2|1.208968617E7|
|              0|1.113673045E7|
|              3|   2652077.88|
|              4|   1863533.79|
|              5|    462094.09|
|              6|    315181.54|
|              8|       895.65|
|              7|       330.75|
|              9|       319.82|
+---------------+-------------+



**7. Average Revenue by Passenger Group**

In [ ]:
clean_df.groupBy(
    'passenger_count'
).agg(
    round(avg('total_amount'), 2).alias('avg_revenue')
).orderBy(
    col('avg_revenue').desc()
).show()

+---------------+-----------+
|passenger_count|avg_revenue|
+---------------+-----------+
|              9|     106.61|
|              7|      82.69|
|              8|      81.42|
|              4|      32.93|
|              2|       30.4|
|              3|      29.78|
|              1|      27.14|
|              6|      26.34|
|              5|      26.08|
|              0|       23.2|
+---------------+-----------+



## **Creation of Customer Summary Behaviour**

In [ ]:
customer_kips = clean_df.agg(
    round(avg('tip_amount'), 2).alias("Average Tip"),
    round(avg('tip_percentage'), 2).alias("Average Tip Percenatage"),
    round(avg('passenger_count'), 2).alias('Average Passenger Count')
)

customer_kips.show()

+-----------+-----------------------+-----------------------+
|Average Tip|Average Tip Percenatage|Average Passenger Count|
+-----------+-----------------------+-----------------------+
|       3.09|                   20.0|                   1.12|
+-----------+-----------------------+-----------------------+



In [ ]:
# Payment Distribution %
payment_distribution.show()

# Average Tip
clean_df.agg(
    round(avg("tip_amount"), 2).alias("avg_tip")
).show()

# Average Tip %
clean_df.agg(
    round(avg("tip_percentage"), 2).alias("avg_tip_percentage")
).show()

# Top Tipping Hours
clean_df.groupBy("pickup_hour") \
    .agg(round(avg("tip_percentage"), 2).alias("avg_tip_percentage")) \
    .orderBy(col("avg_tip_percentage").desc()) \
    .show()

# Passenger Distribution
clean_df.groupBy("passenger_count") \
    .count() \
    .orderBy("passenger_count") \
    .show()

# Revenue by Passenger Count
clean_df.groupBy("passenger_count") \
    .agg(round(sum("total_amount"), 2).alias("revenue")) \
    .orderBy(col("revenue").desc()) \
    .show()

+------------+----------+----------+
|payment_type|trip_count|percentage|
+------------+----------+----------+
|           5|         1|       0.0|
|           1|   2444378|     73.39|
|           3|     15693|      0.47|
|           2|    376318|      11.3|
|           4|     39070|      1.17|
|           0|    455327|     13.67|
+------------+----------+----------+

+-------+
|avg_tip|
+-------+
|   3.09|
+-------+

+------------------+
|avg_tip_percentage|
+------------------+
|              20.0|
+------------------+

+-----------+------------------+
|pickup_hour|avg_tip_percentage|
+-----------+------------------+
|         18|              21.9|
|         15|             21.77|
|         16|             21.69|
|         17|             21.68|
|         19|             21.02|
|         14|             20.84|
|         20|             20.49|
|         10|             20.24|
|         11|             20.13|
|         12|             20.08|
|         21|             20.01|
|         

In [ ]:
clean_df.agg(
    round(avg("tip_percentage"), 2)
    .alias("avg_tip_percentage")
).show()

+------------------+
|avg_tip_percentage|
+------------------+
|              20.0|
+------------------+



In [ ]:
clean_df.groupBy("pickup_hour") \
    .agg(
        round(
            avg("tip_percentage"), 2
        ).alias("avg_tip_percentage")
    ) \
    .orderBy(
        col("avg_tip_percentage").desc()
    ) \
    .show()

+-----------+------------------+
|pickup_hour|avg_tip_percentage|
+-----------+------------------+
|         18|              21.9|
|         15|             21.77|
|         16|             21.69|
|         17|             21.68|
|         19|             21.02|
|         14|             20.84|
|         20|             20.49|
|         10|             20.24|
|         11|             20.13|
|         12|             20.08|
|         21|             20.01|
|         13|             19.97|
|          9|             19.64|
|         23|             18.65|
|         22|             18.56|
|          8|             17.68|
|          3|              17.6|
|          0|             17.21|
|          1|              17.1|
|          2|             17.04|
+-----------+------------------+
only showing top 20 rows


In [ ]:
clean_df.groupBy("passenger_count") \
    .count() \
    .orderBy("passenger_count") \
    .show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|              0| 479978|
|              1|2277734|
|              2| 397726|
|              3|  89056|
|              4|  56591|
|              5|  17719|
|              6|  11965|
|              7|      4|
|              8|     11|
|              9|      3|
+---------------+-------+



In [ ]:
clean_df.groupBy("passenger_count") \
    .agg(
        round(
            sum("total_amount"), 2
        ).alias("revenue")
    ) \
    .orderBy(
        col("revenue").desc()
    ) \
    .show()

+---------------+-------------+
|passenger_count|      revenue|
+---------------+-------------+
|              1|6.181546996E7|
|              2|1.208968617E7|
|              0|1.113673045E7|
|              3|   2652077.88|
|              4|   1863533.79|
|              5|    462094.09|
|              6|    315181.54|
|              8|       895.65|
|              7|       330.75|
|              9|       319.82|
+---------------+-------------+



In [ ]:
clean_df.select(
    "PULocationID",
    "DOLocationID"
).show(10, False)

+------------+------------+
|PULocationID|DOLocationID|
+------------+------------+
|229         |237         |
|236         |237         |
|141         |141         |
|244         |244         |
|244         |116         |
|239         |68          |
|170         |170         |
|234         |148         |
|148         |170         |
|237         |262         |
+------------+------------+
only showing top 10 rows


# **Location Analysis**

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-06-02 05:13:48--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.248.205, 54.230.248.73, 54.230.248.222, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.248.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-06-02 05:13:49 (269 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [ ]:
zone_df = spark.read.csv(
    'taxi_zone_lookup.csv',
    header = True,
    inferSchema = True
)

In [ ]:
zone_df.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [ ]:
zone_df.show(10, False)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows


**1. Join Pickup Zone Information**

In [ ]:
pickup_zone_df = zone_df.select(
    col('LocationID').alias('PULocationID'),
    col('Zone').alias('Pickup_Zone'),
    col('Borough').alias('Pickup_Borough')
)

**2. Join with Taxi Data**

In [ ]:
location_df = clean_df.join(
    pickup_zone_df,
    on = "PULocationID",
    how = 'left'
)

In [ ]:
location_df.select(
    'PULocationID',
    'Pickup_zone',
    'Pickup_Borough'
).show(10, False)

+------------+-----------------------------+--------------+
|PULocationID|Pickup_zone                  |Pickup_Borough|
+------------+-----------------------------+--------------+
|229         |Sutton Place/Turtle Bay North|Manhattan     |
|236         |Upper East Side North        |Manhattan     |
|141         |Lenox Hill West              |Manhattan     |
|244         |Washington Heights South     |Manhattan     |
|244         |Washington Heights South     |Manhattan     |
|239         |Upper West Side South        |Manhattan     |
|170         |Murray Hill                  |Manhattan     |
|234         |Union Sq                     |Manhattan     |
|148         |Lower East Side              |Manhattan     |
|237         |Upper East Side South        |Manhattan     |
+------------+-----------------------------+--------------+
only showing top 10 rows


**3. Top Pickup Zones**

In [ ]:
location_df.groupBy(
    'Pickup_Zone'
).count() \
.orderBy(
    col('count').desc()
).show(20, False)

+----------------------------+------+
|Pickup_Zone                 |count |
+----------------------------+------+
|Midtown Center              |164176|
|Upper East Side South       |159920|
|Upper East Side North       |151777|
|JFK Airport                 |137498|
|Times Sq/Theatre District   |120107|
|Penn Station/Madison Sq West|115455|
|Midtown East                |114472|
|Lincoln Square East         |107336|
|Upper West Side South       |93344 |
|Midtown North               |92881 |
|Union Sq                    |92350 |
|Murray Hill                 |91901 |
|East Chelsea                |87628 |
|LaGuardia Airport           |87401 |
|Clinton East                |80532 |
|Lenox Hill West             |79102 |
|East Village                |77519 |
|West Village                |74177 |
|Midtown South               |72707 |
|Lenox Hill East             |72560 |
+----------------------------+------+
only showing top 20 rows


**4. Top Boroughs by Trips**

In [ ]:
location_df.groupBy(
    "Pickup_Borough"
).count() \
.orderBy(
    col("count").desc()
).show()

+--------------+-------+
|Pickup_Borough|  count|
+--------------+-------+
|     Manhattan|2969051|
|        Queens| 278383|
|      Brooklyn|  59593|
|         Bronx|  13825|
|       Unknown|   8044|
|           N/A|   1300|
|           EWR|    367|
| Staten Island|    224|
+--------------+-------+



**5. Revenue By Borough**

In [ ]:
location_df.groupBy(
    'Pickup_Borough'
).agg(
    round(sum('total_amount'), 2).alias('revenue')
).orderBy(
    col('revenue').desc()
).show()

+--------------+-------------+
|Pickup_Borough|      revenue|
+--------------+-------------+
|     Manhattan|6.776309779E7|
|        Queens|1.991853608E7|
|      Brooklyn|    1785437.9|
|         Bronx|    464710.94|
|       Unknown|    227053.19|
|           N/A|    132059.61|
|           EWR|     36065.78|
| Staten Island|      9358.81|
+--------------+-------------+



**6. Revenue by Pickup Zone**

In [ ]:
location_df.groupBy(
    'Pickup_Zone'
).agg(
    round(sum('total_amount'), 2).alias('revenue')
).orderBy(
    col('revenue').desc()
).show(20, False)

+----------------------------+-------------+
|Pickup_Zone                 |revenue      |
+----------------------------+-------------+
|JFK Airport                 |1.105347922E7|
|LaGuardia Airport           |6694152.28   |
|Midtown Center              |3939425.78   |
|Times Sq/Theatre District   |3238233.4    |
|Upper East Side South       |3184467.59   |
|Upper East Side North       |3087262.71   |
|Penn Station/Madison Sq West|2772810.71   |
|Midtown East                |2706854.19   |
|Lincoln Square East         |2335974.64   |
|Midtown North               |2256626.31   |
|East Chelsea                |2160528.04   |
|Murray Hill                 |2124730.84   |
|Union Sq                    |2015260.66   |
|Upper West Side South       |1982280.76   |
|Clinton East                |1865861.05   |
|East Village                |1724307.68   |
|Midtown South               |1719348.61   |
|West Village                |1637950.94   |
|Lenox Hill East             |1625933.02   |
|Lenox Hil

# **Spark SQL Business Analysis**

**1. Create Temporary View**

In [ ]:
location_df.createOrReplaceTempView(
    'taxi_trips'
)

In [ ]:
spark.sql(
    """
      SELECT * FROM
      taxi_trips
      LIMIT 5
    """
).show()

+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+-----------+----------+------------+--------------+--------------+----------------+--------------------+--------------+
|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_duration_minutes|pickup_hour|pickup_day|pickup_month|pickup_weekday|tip_percentage|revenue_per_mile|         Pickup_Zone|Pickup_Borough|
+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

**1. Top Revenue Hours**

In [ ]:
spark.sql("""
SELECT
  pickup_hour,
  ROUND(SUM(total_amount), 2) AS revenue
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY revenue DESC
LIMIT 10
""").show()

+-----------+----------+
|pickup_hour|   revenue|
+-----------+----------+
|         17|6655982.19|
|         18|6639354.65|
|         16|6136184.58|
|         15|5671291.87|
|         19|5655233.37|
|         14|5358862.61|
|         12|5281467.73|
|         21|5264911.75|
|         20|4999636.19|
|         13|4801315.55|
+-----------+----------+



**2. Top Revenue Weekdays**

In [ ]:
spark.sql("""
SELECT
  pickup_day,
  ROUND(SUM(total_amount), 2) AS revenue
FROM taxi_trips
GROUP BY pickup_day
ORDER BY revenue DESC
""").show()

+----------+----------+
|pickup_day|   revenue|
+----------+----------+
|        16|3640451.19|
|        23|3485788.92|
|        30|3450240.36|
|        31|3353230.47|
|        20|3323849.78|
|        24|3274664.43|
|        15|3229541.66|
|        25|3216764.89|
|        22|3160326.58|
|        17|3156456.03|
|        14|3115970.54|
|         9|3086476.83|
|        29|3076161.61|
|        21|3023352.88|
|        11|2951160.96|
|        10|2930638.44|
|        28|2897376.63|
|         8|2888025.26|
|        18|2832121.41|
|        19|2741046.84|
+----------+----------+
only showing top 20 rows


**3. Top Pickup Zones**

In [ ]:
spark.sql("""
SELECT
  Pickup_Zone,
  COUNT(*) AS trips
FROM taxi_trips
GROUP BY Pickup_Zone
ORDER BY trips DESC
LIMIT 20
""").show(truncate = False)

+----------------------------+------+
|Pickup_Zone                 |trips |
+----------------------------+------+
|Midtown Center              |164176|
|Upper East Side South       |159920|
|Upper East Side North       |151777|
|JFK Airport                 |137498|
|Times Sq/Theatre District   |120107|
|Penn Station/Madison Sq West|115455|
|Midtown East                |114472|
|Lincoln Square East         |107336|
|Upper West Side South       |93344 |
|Midtown North               |92881 |
|Union Sq                    |92350 |
|Murray Hill                 |91901 |
|East Chelsea                |87628 |
|LaGuardia Airport           |87401 |
|Clinton East                |80532 |
|Lenox Hill West             |79102 |
|East Village                |77519 |
|West Village                |74177 |
|Midtown South               |72707 |
|Lenox Hill East             |72560 |
+----------------------------+------+



**4. Revenue by Borough**

In [ ]:
spark.sql("""
SELECT
  Pickup_Borough,
  ROUND(SUM(total_amount), 2) AS revenue
FROM taxi_trips
GROUP BY Pickup_Borough
ORDER BY revenue DESC
""").show(truncate = False)

+--------------+-------------+
|Pickup_Borough|revenue      |
+--------------+-------------+
|Manhattan     |6.776309779E7|
|Queens        |1.991853608E7|
|Brooklyn      |1785437.9    |
|Bronx         |464710.94    |
|Unknown       |227053.19    |
|N/A           |132059.61    |
|EWR           |36065.78     |
|Staten Island |9358.81      |
+--------------+-------------+



**5. Average Tip by Payment Method**

In [ ]:
spark.sql("""
SELECT
  payment_type,
  ROUND(AVG(tip_amount), 2) AS avg_tip
FROM taxi_trips
GROUP BY payment_type
ORDER BY avg_tip DESC
""").show()

+------------+-------+
|payment_type|avg_tip|
+------------+-------+
|           1|   4.11|
|           0|   0.54|
|           3|   0.01|
|           4|   0.01|
|           5|    0.0|
|           2|    0.0|
+------------+-------+



**6.Top Tipping Hours**

In [ ]:
spark.sql("""
SELECT
  pickup_hour,
  ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY avg_tip_percentage DESC
""").show()

+-----------+------------------+
|pickup_hour|avg_tip_percentage|
+-----------+------------------+
|         18|              21.9|
|         15|             21.77|
|         16|             21.69|
|         17|             21.68|
|         19|             21.02|
|         14|             20.84|
|         20|             20.49|
|         10|             20.24|
|         11|             20.13|
|         12|             20.08|
|         21|             20.01|
|         13|             19.97|
|          9|             19.64|
|         23|             18.65|
|         22|             18.56|
|          8|             17.68|
|          3|              17.6|
|          0|             17.21|
|          1|              17.1|
|          2|             17.04|
+-----------+------------------+
only showing top 20 rows


**7. Revenue by Passenger Count**

In [ ]:
spark.sql("""
SELECT
  passenger_count,
  ROUND(SUM(total_amount), 2) AS revenue
FROM taxi_trips
GROUP BY passenger_count
ORDER BY revenue DESC
""").show()

+---------------+-------------+
|passenger_count|      revenue|
+---------------+-------------+
|              1|6.181546996E7|
|              2|1.208968617E7|
|              0|1.113673045E7|
|              3|   2652077.88|
|              4|   1863533.79|
|              5|    462094.09|
|              6|    315181.54|
|              8|       895.65|
|              7|       330.75|
|              9|       319.82|
+---------------+-------------+



In [ ]:
spark.sql("""
SELECT

ROUND(
    SUM(total_amount),
    2
) AS total_revenue,

COUNT(*) AS total_trips,

ROUND(
    AVG(fare_amount),
    2
) AS avg_fare,

ROUND(
    AVG(trip_distance),
    2
) AS avg_distance,

ROUND(
    AVG(trip_duration_minutes),
    2
) AS avg_duration

FROM taxi_trips
""").show()

+-------------+-----------+--------+------------+------------+
|total_revenue|total_trips|avg_fare|avg_distance|avg_duration|
+-------------+-----------+--------+------------+------------+
| 9.03363201E7|    3330787|   18.33|        5.39|       15.04|
+-------------+-----------+--------+------------+------------+



# **Performance Optimization**

**1. Understanding Evaluation**

In [ ]:
df = clean_df.filter(
    col('fare_amount') > 10
)

In [ ]:
df.count()

2160256

In [ ]:
test_df = clean_df.filter(
    col('fare_amount') > 10
)

print('Transformation Created')

Transformation Created


In [ ]:
test_df.count()

2160256

**2. Use Explain()**

In [ ]:
clean_df.explain()

== Physical Plan ==
*(1) Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, coalesce(passenger_count#3L, 0) AS passenger_count#2280L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, cbd_congestion_fee#19, (cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true)) as double) / 60.0) AS trip_duration_minutes#2449, hour(tpep_pickup_datetime#1, Some(Etc/UTC)) AS pickup_hour#2450, dayofmonth(cast(tpep_pickup_datetime#1 as date)) AS pickup_day#2451, month(cast(tpep_pickup_datetime#1 as date)) AS pickup_month#2452, date_format(cast(tpep_pickup_datetime#1 as timestamp), EEEE, Some(Etc/UTC)) AS pickup_weekday#2453, ... 2 more fields]
+- *(1) Filter (((isnotnu

In [ ]:
clean_df.filter(
    col('fare_amount') > 10
).explain() ### What spark plans to do before execution

== Physical Plan ==
*(1) Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, coalesce(passenger_count#3L, 0) AS passenger_count#2280L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, cbd_congestion_fee#19, (cast((unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true)) as double) / 60.0) AS trip_duration_minutes#2449, hour(tpep_pickup_datetime#1, Some(Etc/UTC)) AS pickup_hour#2450, dayofmonth(cast(tpep_pickup_datetime#1 as date)) AS pickup_day#2451, month(cast(tpep_pickup_datetime#1 as date)) AS pickup_month#2452, date_format(cast(tpep_pickup_datetime#1 as timestamp), EEEE, Some(Etc/UTC)) AS pickup_weekday#2453, ... 2 more fields]
+- *(1) Filter ((((isnotn

**3. Cache the data**

In [ ]:
clean_df

## for
# EDA
# KPI Analysis
# Customer Analysis
# Location Analysis

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, trip_duration_minutes: double, pickup_hour: int, pickup_day: int, pickup_month: int, pickup_weekday: string, tip_percentage: double, revenue_per_mile: double]

In [ ]:
clean_df.cache()

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, trip_duration_minutes: double, pickup_hour: int, pickup_day: int, pickup_month: int, pickup_weekday: string, tip_percentage: double, revenue_per_mile: double]

In [ ]:
clean_df.count()

## The first action loads data into memory
## Subsequent operations become faster

3330787

**Without the cache**

*   List item
*   List item





*   Read Data

*   Clean Data
*   Create Features


*   Run Analysis


**With Cache**



*   Read Data
*   Clean Data


*   Create Features
*   Store in memory


*   Run Analysis





**4. Check Cached Data**

In [ ]:
print(
    spark.catalog.isCached(
        'taxi_trips'
    )
)

False


In [ ]:
spark.catalog.cacheTable(
    "taxi_trips"
)

In [ ]:
print(
    spark.catalog.isCached(
        "taxi_trips"
    )
)

True


**5. Repartition**



1. Increase or decrease partitions
2. Full Shuffle
3. More Expensive



In [ ]:
clean_df = clean_df.repartition(8)

**Check current partition**

In [ ]:
clean_df.rdd.getNumPartitions()

8

**6. Coalesce**

1. Usually Decrease Partitions
2. Less Shuffle
3. More Efficient


In [ ]:
clean_df = clean_df.coalesce(2)

#### **Difference between Repartition and Coalesce**

Repartition performs a full shuffle and can increase or decrease partitions.

Coalesce is optimized for reducing partitions with minimal data movement.

**7. Count Rows per Partition**

In [ ]:
partition_sizes = clean_df.rdd.glom().map(
    len
).collect()

print(partition_sizes)

# glom() it groups all elements within each partition into a list

[1665394, 1665393]


**8. Persist (Advanced)**

persist() is used to store (cache) an RDD or DataFrame in memory and/or disk so Spark doesn't have t o recompute it every time you use it

In [ ]:
from pyspark import StorageLevel

clean_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, trip_duration_minutes: double, pickup_hour: int, pickup_day: int, pickup_month: int, pickup_weekday: string, tip_percentage: double, revenue_per_mile: double]

#### **Difference between cache() and persist()**

**cache()**

1. MEMORY_ONLY

**persist()**

1. MEMORY_ONLY
2. MEMORY_AND_DISK
3. DISK_ONLY

**9. Benchmark Example**

In [ ]:
import time

start = time.time()

clean_df.groupBy(
    'pickup_hour'
).count().show()

print(time.time() - start)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|         12|170631|
|         22|172154|
|          1| 60540|
|         13|180774|
|         16|209883|
|          6| 47355|
|          3| 26260|
|         20|186426|
|          5| 21037|
|         19|210807|
|         15|207161|
|         17|242278|
|          9|138889|
|          4| 18527|
|          8|135421|
|         23|128586|
|          7| 97937|
|         10|144706|
|         21|196035|
|         11|156189|
+-----------+------+
only showing top 20 rows
39.00284695625305


In [ ]:
clean_df.write \
.mode('overwrite')\
.parquet('taxi_cleaned_parquet')

In [ ]:
spark.read.parquet(
    'taxi_cleaned_parquet'
).count()

3330787

In [ ]:
clean_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("taxi_cleaned_single_csv")

In [ ]:
import os

os.listdir(
    "taxi_cleaned_single_csv"
)

['part-00000-a890fc55-c554-4dc0-a87e-b6d1d06ec634-c000.csv',
 '._SUCCESS.crc',
 '_SUCCESS',
 '.part-00000-a890fc55-c554-4dc0-a87e-b6d1d06ec634-c000.csv.crc']

In [ ]:
import os

os.listdir(
    "taxi_cleaned_single_csv"
)

['part-00000-a890fc55-c554-4dc0-a87e-b6d1d06ec634-c000.csv',
 '._SUCCESS.crc',
 '_SUCCESS',
 '.part-00000-a890fc55-c554-4dc0-a87e-b6d1d06ec634-c000.csv.crc']

In [ ]:
# from google.colab import files

# files.download(
#     "taxi_cleaned_single_csv/part-00000-180445c3-ba1e-434d-bed3-1643cbac432f-c000.csv"
# )

In [ ]:
location_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("taxi_location_dataset")

In [ ]:
import os

os.listdir("taxi_location_dataset")

['part-00000-d137238b-9fe0-4418-a473-d6173dd68d9e-c000.csv',
 '._SUCCESS.crc',
 '.part-00000-d137238b-9fe0-4418-a473-d6173dd68d9e-c000.csv.crc',
 '_SUCCESS']

In [ ]:
import os

for file in os.listdir("taxi_location_dataset"):
    print(file)

part-00000-d137238b-9fe0-4418-a473-d6173dd68d9e-c000.csv
._SUCCESS.crc
.part-00000-d137238b-9fe0-4418-a473-d6173dd68d9e-c000.csv.crc
_SUCCESS


In [ ]:
from google.colab import files

files.download(
    "taxi_location_dataset/part-00000-d137238b-9fe0-4418-a473-d6173dd68d9e-c000.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls -lh taxi_location_dataset

total 576M
-rw-r--r-- 1 root root 575M Jun  2 05:20 part-00000-d137238b-9fe0-4418-a473-d6173dd68d9e-c000.csv
-rw-r--r-- 1 root root    0 Jun  2 05:20 _SUCCESS


In [ ]:
clean_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("taxi_cleaned_single_csv")